# 01 — Getting started with groundline

groundline builds a queryable **fact base** of a Fortran codebase from flang's
semantic parse-tree dumps (`flang -fc1 -fdebug-dump-parse-tree`). The pipeline:

```
flang dump (text)  →  frontend: extract(sources) → IR  →  consumers
                                                            ├─ ParseForest  (NetworkX graphs)
                                                            ├─ graph_view   (renderable elements)
                                                            └─ Explorer     (Jupyter widget)
```

The **IR** (`groundline/ir.py`) is the seam: a small collection of entities
(interned atoms) and relations (tuple-sets) over them. Everything flang-specific
lives *below* the seam, in `groundline/frontend/`; consumers — including every
notebook in this directory — import only `groundline.{ir, parse_forest,
graph_view, explorer}` and never learn that flang exists. That is what makes a
future frontend swap (e.g. LFortran) a localized change (see `docs/DESIGN.md`).

**This notebook is portable**: it runs anywhere the repo is checked out. It uses
the small Fortran fixtures committed under `tests/f90/` — no corpus access and no
flang installation needed. Notebooks 02–04 apply the same ideas to a real
458-file MOM6 + FMS2 corpus.

What you will see here:

1. building a `ParseForest` from parse-tree dumps,
2. the IR's universe — entities with **scope-qualified identity**,
3. its relations — `uses`, `contains`, interface membership,
4. the **call relation, stratified by confidence** (`resolved` / `assumed` /
   `unresolved`) and the derived **may/must** views,
5. that the IR is plain data a frontend merely populates,
6. the NetworkX graphs and a first look at the interactive Explorer.

## Parameters

The one cell to edit. Each fixture is a minimal, self-contained Fortran program
pinning one construct (they double as the conformance corpus, `docs/VISION.md`
D7); the four picked here each carry one concept this notebook teaches.

In [ ]:
from pathlib import Path

# tests/f90 ships with the repo; this resolves relative to the notebooks/ dir.
FIXTURES_DIR = (Path.cwd() / ".." / "tests" / "f90").resolve()
assert FIXTURES_DIR.is_dir(), (
    f"{FIXTURES_DIR} not found — launch Jupyter from the groundline repo "
    "so the notebook runs inside notebooks/."
)

FIXTURES = [
    "test_name_collision",     # scope-qualified identity: three modules, one name
    "test_interface_basic",    # a generic interface and its specific procedures
    "test_type_bound_generic", # a derived type with type-bound procedures
    "test_external_calls",     # calls whose targets are defined nowhere (unresolved)
]
paths = [FIXTURES_DIR / f"{name}_ptree" for name in FIXTURES]
[p.name for p in paths]

## Build a forest

`ParseForest(paths)` hands the paths to the default frontend
(`FlangDumpFrontend`), which parses the dumps and projects them onto an
`IR`. Files the frontend cannot parse are collected in `ir.file_errors`
and skipped — one bad file never aborts the forest.

You can also pass a pre-built IR directly (`ParseForest(ir=...)`); we use that
later to show the seam is real.

In [ ]:
from groundline.parse_forest import ParseForest

forest = ParseForest(paths)
ir = forest.ir
print(f"{len(ir.entities)} entities, {len(ir.file_errors)} file errors")

## The universe: entities with scope-qualified identity

Entity *kinds* — module, subroutine, function, interface, derived type — are the
unary relations (sets) of the universe. Every entity is identified by a
scope-qualified `EntityId` string such as `collide_a_mod::apply_bc`, **never a
bare name** (design principle #7).

In [ ]:
for kind_view in ("modules", "subroutines", "functions", "interfaces", "derived_types"):
    entities = getattr(ir, kind_view)
    print(f"{kind_view:>14}: {len(entities):2}   e.g. {sorted(e.id for e in entities)[:3]}")

### Why identity matters: the name-collision fixture

`test_name_collision` defines `apply_bc` in **three different modules** with
identical signatures — the name is the only thing they share. A name-keyed
consumer would silently merge them into one node (this was weakness W5, fixed
and pinned in Phase 3). In the IR they are three distinct atoms, and the caller's
three calls resolve to the right one each time.

In [ ]:
collisions = [e for e in ir.subroutines if e.name == "apply_bc"]
for e in sorted(collisions, key=lambda e: e.id):
    print(f"id={e.id!r:32} name={e.name!r} scope={e.scope!r} defined={e.defined}")

# Lookup is by id, never by name:
ir.get("collide_b_mod::apply_bc")

## Relations: `uses`, `contains`, interface membership

Structural facts are relations between atoms. A `Use` edge records the
importing scope, the module name, and any only-list / renames — the caller in
the collision fixture deliberately reaches each `apply_bc` through a different
USE form (wildcard-with-rename, only-list, only-list-with-rename).

In [ ]:
for u in sorted(ir.uses, key=lambda u: (u.scope, u.module)):
    if u.scope == "collide_caller_mod":
        print(f"USE {u.module:15} only={u.only!r:18} renames={u.renames!r}")

A generic interface is an entity whose `interface_members` relation points at
its specific procedures, and a derived type carries its type-bound `bindings`
(binding name → implementation name) as entity facts:

In [ ]:
iface = ir.get("interface_basic_mod::compute")
print(f"{iface.id} -> {sorted(m.id for m in ir.members(iface.id))}")

gadget = ir.get("tbp_mod::gadget_t")
print(f"{gadget.id} bindings: {gadget.bindings}")

## The call relation, stratified by confidence

The heart of the IR (D3 in `docs/VISION.md`): the call relation is stored as
**three pure relations**, not one relation with a tag —

* `calls_resolved` — the target identity is certain (read from flang sema's
  resolution, or a direct call to a unique visible procedure);
* `calls_assumed` — the target is a guess (genuine dynamic dispatch: the
  declared type's binding, though an override may win at runtime);
* `calls_unresolved` — the callee is known to exist but defined nowhere in the
  parsed set; the target is a **first-class entity with `defined=False`**,
  never silently dropped.

In `test_external_calls`, `ext_sub`/`ext_fun` are declared `EXTERNAL` with no
definition anywhere, so those edges land in `calls_unresolved` — while the
`helper_sub` call in the same routine stays `resolved`.

In [ ]:
print("resolved:")
for caller, callee in sorted(ir.calls_resolved):
    print(f"  {caller}  ->  {callee}")
print("assumed:", sorted(ir.calls_assumed))
print("unresolved:")
for caller, callee in sorted(ir.calls_unresolved):
    e = ir.get(callee)
    print(f"  {caller}  ->  {callee}   (kind={e.kind}, defined={e.defined})")

Two things worth noticing above:

* The generic call in `test_interface_basic` produced *resolved* edges to the
  specific procedures (`compute_real`, `compute_int`, `compute_logical`) — sema
  already resolved each call site, so there is no "fan out to every member"
  guessing.
* `assumed` is empty. Only **genuine dynamic dispatch** (a polymorphic receiver
  whose override is unknowable statically) produces it, and no self-contained
  fixture exercises that construct yet (a recorded gap in
  `tests/f90/MANIFEST.md`). The real corpus in notebook 04 has such edges; we
  also build one by hand below.

### The may/must lattice

The strata hand us the standard sound-analysis lattice as computed views:

* `ir.calls_must` = `calls_resolved` — the **under-approximation**: an edge here
  definitely exists, so a violation found on it is a *definite* finding.
* `ir.calls` (may) = union of all three strata — the **over-approximation**: a
  violation found only here is *possible*.

`ir.call_confidence(caller, callee)` answers which stratum a single edge came
from (or `None` if it is not a call edge).

In [ ]:
print(f"may  = {len(ir.calls)} edges")
print(f"must = {len(ir.calls_must)} edges")
print(ir.call_confidence("collide_caller_mod::drive", "collide_a_mod::apply_bc"))
print(ir.call_confidence("ext_caller_mod::test_external_calls", "ext_sub"))

## The IR is just data (the seam is real)

Nothing about the IR requires flang: it is a plain dataclass of entities and
tuple-sets that any frontend could populate. To prove it — and to demonstrate
the `assumed` stratum the fixtures lack — here is a tiny hand-built IR with one
dynamic-dispatch-style edge, consumed by the very same `ParseForest`:

In [ ]:
from groundline.ir import IR, Entity, MODULE, SUBROUTINE

tiny = IR()
for e in (
    Entity(id="m", kind=MODULE, name="m"),
    Entity(id="m::caller", kind=SUBROUTINE, name="caller", scope="m"),
    Entity(id="m::step_impl", kind=SUBROUTINE, name="step_impl", scope="m"),
):
    tiny.entities[e.id] = e
tiny.contains |= {("m", "m::caller"), ("m", "m::step_impl")}
# e.g. `call obj%step()` on a polymorphic receiver: the declared type's binding
# is the best static answer, so the frontend records the edge as *assumed*.
tiny.calls_assumed.add(("m::caller", "m::step_impl"))

hand_forest = ParseForest(ir=tiny)
print(hand_forest.ir.call_confidence("m::caller", "m::step_impl"))
print(f"may={len(tiny.calls)} must={len(tiny.calls_must)}  (assumed edges are may-only)")

## Graphs

`ParseForest` builds NetworkX graphs from the relations. Every call-graph edge
carries its stratum as a `confidence` attribute, and `must_only=True` keeps the
compiler-certain subgraph (edges only — `defined=False` targets remain as
isolated nodes, so the node set is stable across the two views).

In [ ]:
g_may = forest.get_call_graph()
g_must = forest.get_call_graph(must_only=True)
print(f"may:  {g_may.number_of_nodes()} nodes, {g_may.number_of_edges()} edges")
print(f"must: {g_must.number_of_nodes()} nodes, {g_must.number_of_edges()} edges")
for u, v, d in sorted(g_may.edges(data=True), key=lambda e: (e[0].id, e[1].id)):
    if d["confidence"] != "resolved":
        print(f"  non-must edge: {u.id} -> {v.id}  [{d['confidence']}]")

The module dependency graph has module-name nodes and an edge for every USE
(lifted from contained scopes to the enclosing module) plus derived-type
EXTENDS relationships. Notebook 03 builds real analyses on it.

In [ ]:
g_mod = forest.get_module_dependency_graph()
sorted(g_mod.edges())

## A first look at the Explorer

The interactive widget over the same facts. Pick a category and an entity to see
its neighbourhood; the legend decodes the visual encoding — edge **line style**
is the confidence stratum (solid resolved, dashed assumed, dotted unresolved),
edge **colour** is direction relative to the selection, and ghosted nodes are
referenced-but-never-parsed (`defined=False`) targets such as `ext_sub`.

(The widget needs a live kernel — re-run this notebook in Jupyter to interact.
Under headless execution it just renders an empty shell.)

In [ ]:
from groundline.explorer import Explorer

Explorer(forest)

## Where to go next

* **02_explore_corpus** — the Explorer and programmatic neighbourhood queries
  over the real 458-file MOM6 + FMS2 corpus.
* **03_module_dependencies** — module-graph analyses: fan-in/out, cycles,
  which parts of FMS2 MOM6 actually needs.
* **04_confidence_queries** — the may/must lattice as a query substrate:
  must-vs-may reachability, unresolved-target censuses, and a taste of the
  invariant checks the planned query layer (Phase 5) will formalize.

Notebooks 02–04 need the corpus on glade (or set `GROUNDLINE_CORPUS`); see
`notebooks/README.md`.